# ARC-AGI-3 Solver — Qwen3.8-27B-FP8

This notebook runs the **TAAF ARC-AGI-3 solver** using a locally mounted **Qwen3.8-27B-FP8** checkpoint through an OpenAI-compatible **vLLM** inference server.

## Model

- **Model:** `Qwen/Qwen3.8-27B-FP8`
- **Format:** Hugging Face / Safetensors
- **Quantization:** FP8
- **Kaggle Model:** `foysalemonshanto/qwen3-8-27b-fp8-repacked-v1`
- **Variation:** `hf-fp8`
- **Version:** `1`
- **Served model ID:** `Qwen/Qwen3.8-27B-FP8`

### Kaggle model path

```text
/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1

In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
# Fail-fast GPU assert: metadata machine_shape + --accelerator alone can still
# bind P100 (3 wasted pushes on serving-lab proved it; the competition source
# attachment is the real RTX Pro 6000 gate). Die here, before any setup cost.
import subprocess as _sp

_gpu = _sp.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("boot gpu:", (_gpu.stdout or "").strip() or (_gpu.stderr or "").strip())
_gpu_name = (_gpu.stdout or "").upper()
assert "RTX" in _gpu_name and "6000" in _gpu_name, (
    f"GPU misbind — expected RTX Pro 6000, got: {_gpu.stdout!r} {_gpu.stderr!r}"
)


In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# Boot attestation (doctrine v2, 2026-08-17): the mounted weights must be the
# OFFICIAL Qwen3.8-FP8. Discriminators verified offline against both the
# official HF snapshot and the vrfai 3.6 config: 3.8 = quant_method fp8 /
# fmt e4m3 / transformers 5.8.0.dev0; 3.6-vrfai = compressed-tensors
# config_groups / transformers 5.6.2. A wrong mount must DIE here, before any
# game action is spent. --served-model-name is a rename and proves nothing.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
_q = _cfg.get("quantization_config") or {}
assert _cfg.get("architectures") == ["Qwen3_5ForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _q.get("quant_method") == "fp8" and _q.get("fmt") == "e4m3", (
    f"attest FAIL: quantization_config is not official fp8/e4m3: {_q}")
assert _cfg.get("transformers_version") == "5.8.0.dev0", (
    f"attest FAIL: transformers_version {_cfg.get('transformers_version')} "
    "(vrfai 3.6 stamps 5.6.2)")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_idx_path = QWEN_MODEL_PATH / "model.safetensors.index.json"
if _idx_path.is_file():
    print("attest: index sha256", _hashlib.sha256(_idx_path.read_bytes()).hexdigest())
_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
assert _shards, "attest FAIL: no safetensors shards at model path"
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert _total > 25_000_000_000, f"attest FAIL: total shard bytes {_total} too small for 27B FP8"
_h = _hashlib.sha256()
with open(_shards[0], "rb") as _f:
    _h.update(_f.read(1 << 20))
print("attest: first-shard-1MiB sha256", _h.hexdigest())

# Greedy decode fingerprint — logged (not asserted) for cross-run comparison.
_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
if not _base.endswith("/v1"):
    _base += "/v1"
_body = json.dumps({
    "model": QWEN_SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly the sum of 17 and 25, then the word quack."}],
    "temperature": 0.0,
    "max_tokens": 48,
    "chat_template_kwargs": {"enable_thinking": False},
}).encode()
_req = _rq.Request(_base + "/chat/completions", data=_body, headers={
    "Content-Type": "application/json",
    "Authorization": "Bearer " + (os.environ.get("LOCAL_ANALYZER_API_KEY") or "EMPTY"),
})
with _rq.urlopen(_req, timeout=180) as _resp:
    _reply = json.loads(_resp.read())["choices"][0]["message"].get("content") or ""
print("attest: decode fingerprint", repr(_reply)[:160])
print("attest: decode sha256", _hashlib.sha256(_reply.encode()).hexdigest())
print("attest: OK — official Qwen3.8-FP8 signature verified before any game")


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Smoke/eval hook: a NORMAL COMMIT runs a 4-game offline smoke on the scored
# GPU class — the yield_carryover ladder read (on top of the effort baseline).
# The scored rerun path (KAGGLE_IS_COMPETITION_RERUN) never enters this branch.
# Games identical to effort-smoke: vc33/tn36 carry effort-smoke comparators
# (L2/4.73, L1/1.52); sk48 is the thinking-stall game; ft09 the digest-lever game.
SMOKE_GAMES = ["vc33-5430563c", "tn36-ef4dde99", "ft09-0d8bbf25", "sk48-d8078629"]

if not run_as_submission:
    import arc_agi
    from taaf.game_api import ArcadeSpec, GameAPI

    def _resolve_env_dir():
        candidates = [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files"),
        ]
        for cand in candidates:
            if cand.is_dir():
                return str(cand)
        for hit in Path("/kaggle/input").rglob("environment_files"):
            if hit.is_dir():
                return str(hit)
        raise RuntimeError("environment_files dir not found in /kaggle/input")

    _env_dir = _resolve_env_dir()
    _spec = ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=_env_dir)
    bm.games = [GameAPI(env_name=name, arcade_spec=_spec) for name in SMOKE_GAMES]
    bm.n_passes = 1
    bm.game_weights = None
    bm.label = "carryover-smoke"
    # Per-game cap ~50 min (bundled solver config: 7920 s). Games run
    # concurrently (solver concurrency 28), so wall = setup + ~50 min + teardown.
    bm.solver.max_runtime_s_per_game = 3000.0
    # Global backstop 3.5 h from notebook start => total run <= 4 h.
    soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=12600)
    print(f"smoke hook: {len(bm.games)} games, env_dir={_env_dir}, "
          f"per_game_cap={bm.solver.max_runtime_s_per_game}s, soft_end={soft_end}")
else:
    print("scored rerun: smoke hook inert — full competition games")

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))


In [ ]:
# Graft install — TWO grafts, stated explicitly (ladder arm, not single-variable
# vs stock: the variable is carryover, measured ON TOP of the effort baseline).
# 1. EFFORT_MEDIUM=1 (graft_effort; EFFORT_DEAD_RETRY=0) — the validated
#    baseline riding tonight's arm (effort-smoke 08-23: clean, 0 dead, 217/217
#    medium payloads, vc33 L2 / tn36 L1).
# 2. YIELD_CARRYOVER=1 (graft_carryover; YIELD_SLICE_CAP=0) — THE ladder
#    variable of this smoke: slice-digest carry-forward only, cap disabled
#    (digest-only first per the smoke plan; the cap smokes later).
os.environ["EFFORT_MEDIUM"] = "1"
os.environ["EFFORT_DEAD_RETRY"] = "0"
os.environ["YIELD_CARRYOVER"] = "1"
os.environ["YIELD_SLICE_CAP"] = "0"

_EFFORT_SOURCE = '"""Effort-medium graft — reasoning_effort=medium on every chat request +\ndead-completion retry hardening.\n\nMotivation (docs/RESEARCH-2026-08-21-bug-lever-hunt.md Tier-1 #2, as amended\nby the wave-2 CORRECTIONS: shipping arms already run temp 0.6/top_p 0.95/\ntop_k 20, so sampling is NOT touched here — the surviving lever is\nreasoning_effort only):\n- The official chat template defaults ``reasoning_effort`` to **xhigh**\n  (verified by rendering our snapshot\'s template); the harness sends only\n  ``enable_thinking``, so every scored call carries xhigh with\n  max_tokens=None.\n- Measured live symptom: **122 thinking-only dead completions**\n  (finish_reason=stop, zero tool calls, 111 with zero content) = 2.62M\n  reasoning chars ~= 650-750k tokens ~= ~6h of decode producing nothing.\n- Paired hardening (same cluster): when a completion returns the dead\n  signature, the NEXT request in the slice forces ``tool_choice`` to the\n  ``python`` function and appends one user line:\n  "Your previous reasoning produced no action — act now."\n\nSeams (verified against the June stock tree,\nscratchpad/bundles/june_stock/src/ARC3-Inference; tool_agent.py md5\n7fea036d7d366b8a07dafa6d8e39a821):\n- openai_compat.py:64-68 — the vllm branch sets\n  ``payload["chat_template_kwargs"] = {"enable_thinking": bool(thinking)}``;\n  the wrapper adds ``reasoning_effort`` (setdefault: an explicit future\n  value always wins) into that same dict, so the key rides only on requests\n  that already carry chat_template_kwargs (vllm requests — the scored path).\n- tool_agent.py:34 — ``build_chat_payload`` is imported BY NAME into\n  tool_agent (and tools/chat.py:10, a dev CLI) => dual-namespace rebinding:\n  the wrapper is bound into openai_compat AND tool_agent (and any already-\n  imported ``inference.tools.chat``).\n- tool_agent.py:1282-1301 — ``ToolAgent._chat_completion`` builds the\n  payload; ``tool_choice`` comes from ``_request_tool_choice(tools)``\n  (tool_agent.py:311-312, always "auto"), with no tool_choice parameter on\n  ``_chat_completion`` itself — hence the force mechanism: the patched\n  ``_chat_completion`` sets a thread-local flag around the inner call and\n  the patched ``_request_tool_choice`` returns the named-function choice\n  while it is set (games run concurrently in solver threads; thread-local\n  keeps arms independent). The only ``_request_tool_choice`` call inside\n  that window is the payload build at tool_agent.py:1299 (the other call\n  sites, :1605 and :1789, run outside the window).\n- tool_agent.py:1854-1856 + :1894-1927 — the dead-completion signature and\n  the stock no-tool-call retry loop this hardens: on\n  finish_reason=="stop" with zero tool calls and empty content (and no\n  ``<tool_call>`` markup, which stock recovery at :1859-1861 handles\n  itself), the next request is forced. The injected user line exists only\n  in the wire request (the loop\'s own ``messages`` list is not mutated),\n  and pending state is cleared at every ``analyze`` entry so the force\n  never leaks across slices.\n\nFail-open invariants:\n- Inner calls are never wrapped in try/except — crashes propagate as stock.\n- All graft logic sits in blanket try/except; any error => stock behavior.\n- EFFORT_MEDIUM=0 disables the kwargs injection; EFFORT_DEAD_RETRY=0\n  disables the retry hardening — each checked at call time; both off at\n  install time => SKIP (no patch applied).\n- EFFORT_LEVEL overrides the injected level (default "medium").\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport sys\nimport threading\nfrom typing import Any\n\nACT_NOW_LINE = "Your previous reasoning produced no action — act now."\nFORCED_TOOL_CHOICE = {"type": "function", "function": {"name": "python"}}\n\n_tls = threading.local()\n\n\ndef _flag_enabled(name: str) -> bool:\n    return os.environ.get(name, "1").strip() not in {"0", "false", "False"}\n\n\ndef _effort_enabled() -> bool:\n    return _flag_enabled("EFFORT_MEDIUM")\n\n\ndef _dead_retry_enabled() -> bool:\n    return _flag_enabled("EFFORT_DEAD_RETRY")\n\n\ndef _effort_level() -> str:\n    return os.environ.get("EFFORT_LEVEL", "medium").strip() or "medium"\n\n\ndef _is_dead_completion(result: Any, agent_mod: Any) -> bool:\n    """The measured signature: finish_reason=stop, zero tool calls, empty\n    content. Completions carrying <tool_call> markup are NOT dead — stock\n    markup recovery (tool_agent.py:1859-1861) owns those."""\n    try:\n        finish_reason = str(getattr(result, "finish_reason", "") or "")\n        if finish_reason != "stop":\n            return False\n        message = getattr(result, "message", None)\n        if not isinstance(message, dict):\n            return False\n        if message.get("tool_calls"):\n            return False\n        normalize = getattr(agent_mod, "_normalize_message_content", None)\n        raw_content = message.get("content", "")\n        content = normalize(raw_content) if callable(normalize) else str(raw_content or "")\n        if str(content or "").strip():\n            return False\n        extract = getattr(agent_mod, "_extract_reasoning_text", None)\n        reasoning = extract(message) if callable(extract) else ""\n        has_markup = getattr(agent_mod, "_contains_tool_call_markup", None)\n        if callable(has_markup) and has_markup(str(reasoning or ""), str(content or "")):\n            return False\n        return True\n    except Exception:  # noqa: BLE001 — unparseable result => not dead\n        return False\n\n\ndef install() -> str:\n    if not _effort_enabled() and not _dead_retry_enabled():\n        return "effort_medium: SKIP (EFFORT_MEDIUM=0 and EFFORT_DEAD_RETRY=0)"\n    try:\n        from inference.utils import openai_compat as compat_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"effort_medium: SKIP (openai_compat module missing: {exc!r})"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"effort_medium: SKIP (tool_agent module missing: {exc!r})"\n\n    original_build = getattr(compat_mod, "build_chat_payload", None)\n    if original_build is None:\n        return "effort_medium: SKIP (build_chat_payload missing)"\n    if getattr(agent_mod, "build_chat_payload", None) is None:\n        return "effort_medium: SKIP (tool_agent.build_chat_payload rebind seam missing)"\n    tool_agent_cls = getattr(agent_mod, "ToolAgent", None)\n    if tool_agent_cls is None:\n        return "effort_medium: SKIP (missing ToolAgent)"\n    original_chat = getattr(tool_agent_cls, "_chat_completion", None)\n    if original_chat is None:\n        return "effort_medium: SKIP (ToolAgent._chat_completion missing)"\n    original_tool_choice = getattr(agent_mod, "_request_tool_choice", None)\n    if original_tool_choice is None:\n        return "effort_medium: SKIP (_request_tool_choice missing — force seam moved)"\n    original_analyze = getattr(tool_agent_cls, "analyze", None)\n    if original_analyze is None:\n        return "effort_medium: SKIP (ToolAgent.analyze missing)"\n    if getattr(compat_mod.build_chat_payload, "_effort_medium_patched", False):\n        return "effort_medium: SKIP (already applied)"\n\n    # --- 1. reasoning_effort alongside enable_thinking (openai_compat.py:68) ---\n\n    def build_payload_with_effort(*args: Any, **kwargs: Any) -> dict[str, Any]:\n        payload = original_build(*args, **kwargs)\n        try:\n            if _effort_enabled():\n                template_kwargs = payload.get("chat_template_kwargs")\n                if isinstance(template_kwargs, dict) and "enable_thinking" in template_kwargs:\n                    template_kwargs.setdefault("reasoning_effort", _effort_level())\n        except Exception:  # noqa: BLE001 — payload shape drift => stock payload\n            pass\n        return payload\n\n    # --- 2. forced tool_choice while a dead-retry request is in flight ---\n\n    def request_tool_choice_with_force(tools: Any) -> Any:\n        try:\n            if tools and getattr(_tls, "force_python", False):\n                return dict(FORCED_TOOL_CHOICE)\n        except Exception:  # noqa: BLE001\n            pass\n        return original_tool_choice(tools)\n\n    # --- 3. dead-completion detection + hardened retry request ---\n\n    def chat_completion_with_dead_retry(self: Any, messages: Any, *args: Any, **kwargs: Any) -> Any:\n        forced = False\n        try:\n            if _dead_retry_enabled() and getattr(self, "_eff_dead_pending", False):\n                self._eff_dead_pending = False\n                messages = list(messages) + [{"role": "user", "content": ACT_NOW_LINE}]\n                _tls.force_python = True\n                forced = True\n        except Exception:  # noqa: BLE001\n            forced = False\n        try:\n            # Never guard the inner call: crashes/RequestExceptions are stock.\n            result = original_chat(self, messages, *args, **kwargs)\n        finally:\n            if forced:\n                try:\n                    _tls.force_python = False\n                except Exception:  # noqa: BLE001\n                    pass\n        try:\n            if _dead_retry_enabled():\n                self._eff_dead_pending = _is_dead_completion(result, agent_mod)\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    # --- 4. pending state never leaks across slices ---\n\n    def analyze_with_clear(self: Any, *args: Any, **kwargs: Any) -> Any:\n        try:\n            self._eff_dead_pending = False\n        except Exception:  # noqa: BLE001\n            pass\n        return original_analyze(self, *args, **kwargs)\n\n    build_payload_with_effort._effort_medium_patched = True  # type: ignore[attr-defined]\n    compat_mod.build_chat_payload = build_payload_with_effort\n    agent_mod.build_chat_payload = build_payload_with_effort  # by-name import, tool_agent.py:34\n    chat_cli = sys.modules.get("inference.tools.chat")\n    if chat_cli is not None and getattr(chat_cli, "build_chat_payload", None) is not None:\n        chat_cli.build_chat_payload = build_payload_with_effort  # tools/chat.py:10 (dev CLI)\n    agent_mod._request_tool_choice = request_tool_choice_with_force\n    tool_agent_cls._chat_completion = chat_completion_with_dead_retry\n    tool_agent_cls.analyze = analyze_with_clear\n    install.originals = {  # type: ignore[attr-defined]\n        "build_chat_payload": original_build,\n        "_request_tool_choice": original_tool_choice,\n        "_chat_completion": original_chat,\n        "analyze": original_analyze,\n    }\n    return "effort_medium: OK"\n'
_CARRYOVER_SOURCE = '"""Yield-carryover graft — slice-digest carry-forward + slice cap on yield-resume.\n\nMotivation (live-measured, docs/RESEARCH-2026-08-21-bug-lever-hunt.md Tier-1 #1):\n2,234 turn-slices audited; 43.3% of slices ended with NO env action\n("Yielded control to solver: turn_time_budget") and 51.5% of ALL wall\n(130,631s/253,830s) elapsed in slices that ended without an action. On\nyield-resume the turn conversation is rebuilt with only ~2-4 messages\n(history_messages 14->11->4->2 inside one turn) — every tool result and all\nreasoning from the previous slice is discarded, the model re-grounds from\nzero, and 71 byte-identical python snippets were re-issued within single\nturns (worst turn: 35 slices; 18 slices/3,584s/118k tokens for ONE action).\n\nTwo mechanisms, separately flagged:\n\n1. YIELD_CARRYOVER (default on): when a slice ends yielded-without-action,\n   build a bounded structured digest of that slice — every tool call\'s code\n   (first ~200 chars) + its result (first ~400 chars) + the last assistant\n   reasoning tail (~500 chars) — and inject it exactly once as a block inside\n   the resumed slice\'s initial user prompt ("Previous slice this turn already\n   ran ..."). Empty yielded slices (yield fired before any request) keep the\n   previous slice\'s digest instead of erasing it.\n2. YIELD_SLICE_CAP (default "3", 0 disables): counts slices per turn\n   (turn = same ``analysis_step`` re-entered after yielded_control, the\n   solver\'s resume contract). From slice 3 onward the resumed slice\'s user\n   prompt is prefixed with a FINAL-SLICE instruction: no further exploratory\n   analysis, the response must end in an ``action(...)`` call. The cap is\n   behavioral (an instruction, not a synthetic action) — the harness never\n   fabricates actions on the model\'s behalf.\n\nSeams (verified against the June stock tree,\nscratchpad/bundles/june_stock/src/ARC3-Inference; tool_agent.py md5\n7fea036d7d366b8a07dafa6d8e39a821, solver.py md5\n7c2840743245402a55f8f62485752b5e — byte-identical to the public\njeroencottaar June source share and the banking_v22 bundle):\n\n- solver.py:311-322 — the resume contract this graft keys on:\n  ``if getattr(result, "yielded_control", False): retry_analysis_step =\n  analysis_step; continue`` — the re-entry calls ``analyze`` with the SAME\n  ``analysis_step`` (chosen at solver.py:284-288, passed at solver.py:299).\n- tool_agent.py:1777-1778 — the yield trigger ("turn_time_budget" once\n  ``self._yield_seconds`` elapses in a slice).\n- tool_agent.py:2015-2019 — the context wipe: ``finally:`` rebinds\n  ``self._history_messages = self._persistent_history_messages(messages,...)``\n  where ``messages`` is the FULL slice conversation. That call is our digest\n  source: the patched ``_persistent_history_messages`` stashes ``messages``\n  before delegating, so the digest sees everything the trim throws away\n  (tool_agent.py:1653-1670 keeps <=30 assistant turns, then the token trim at\n  tool_agent.py:1672-1690 under LOCAL_ANALYZER_CONTEXT_WINDOW=32768 produces\n  the observed 2-4 survivors).\n- tool_agent.py:1727-1733 — the slice\'s initial user prompt is built once per\n  ``analyze`` call by ``_build_user_prompt`` (def at tool_agent.py:1161);\n  the patched ``_build_user_prompt`` records its first 80 chars as the\n  slice-boundary marker (the slice\'s own messages are everything after the\n  last user message starting with it) and applies any pending injection —\n  hence "injected exactly once".\n- tool_agent.py:1706-1717 — ``ToolAgent.analyze`` signature\n  (``analysis_step`` keyword at :1713); the wrapper\'s turn key is\n  ``(str(state_path), analysis_step)``.\n- tool_agent.py:2059-2062 / dataclass at :369-374 — ``AnalyzerTurnResult``\n  carries ``yielded_control`` + ``step_executed``, the post-call signal for\n  "this slice yielded without acting".\n\nEnvelope note: see ENVELOPE.md — neither mechanism can extend run duration\n(the digest only adds bounded prompt tokens; the cap only shortens turns).\n\nFail-open invariants:\n- The inner ``analyze`` / ``_persistent_history_messages`` /\n  ``_build_user_prompt`` calls are never wrapped in try/except — an inner\n  crash propagates exactly as stock.\n- All graft logic (turn tracking, digest build, injection) sits inside\n  blanket try/except; any error means "no injection, stock behavior".\n- Both flags checked at call time: flags off => the wrappers are pure\n  pass-throughs (byte-identical prompts and results).\n- install() presence-gates every seam symbol and fails toward stock.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom typing import Any\n\nDIGEST_CODE_CHARS = 200\nDIGEST_RESULT_CHARS = 400\nDIGEST_REASONING_CHARS = 500\nDIGEST_MAX_ENTRIES = 10\nDIGEST_MAX_CHARS = 6000\n_HEAD_MARKER_CHARS = 80\n\nDIGEST_HEADER = "Previous slice this turn already ran (carried over so you do not repeat work):"\nDIGEST_FOOTER = (\n    "Do not re-run identical inspections; build on these carried-over results and move toward an action."\n)\nFINAL_SLICE_INSTRUCTION = (\n    "FINAL SLICE FOR THIS TURN: you have used the allowed analysis slices for this turn without acting. "\n    "Do not run further exploratory analysis. Respond with a single `python` tool call whose code ends by "\n    "calling `action(actions)` with the best valid action or ordered batch you currently have."\n)\n\n\ndef _flag_enabled(name: str, default: str = "1") -> bool:\n    return os.environ.get(name, default).strip() not in {"0", "false", "False"}\n\n\ndef _carryover_enabled() -> bool:\n    return _flag_enabled("YIELD_CARRYOVER")\n\n\ndef _slice_cap() -> int:\n    """Configured cap (slices per turn); 0 disables the cap mechanism."""\n    raw = os.environ.get("YIELD_SLICE_CAP", "3").strip()\n    if raw in {"", "false", "False"}:\n        return 0\n    try:\n        return max(0, int(raw))\n    except ValueError:\n        return 3\n\n\ndef _truncate(text: str, limit: int) -> str:\n    text = str(text)\n    if len(text) <= limit:\n        return text\n    return f"{text[:limit].rstrip()} ...[+{len(text) - limit} chars]"\n\n\ndef _message_text(message: dict[str, Any]) -> str:\n    """Text of a message whose content is either a string or a parts list."""\n    content = message.get("content")\n    if isinstance(content, str):\n        return content\n    if isinstance(content, list):\n        parts = []\n        for part in content:\n            if isinstance(part, dict) and part.get("type") == "text":\n                parts.append(str(part.get("text", "")))\n        return "\\n".join(parts)\n    return ""\n\n\ndef _tool_call_code(tool_call: Any) -> str:\n    function = tool_call.get("function", {}) if isinstance(tool_call, dict) else {}\n    raw_arguments = function.get("arguments", "")\n    if isinstance(raw_arguments, str):\n        try:\n            parsed = json.loads(raw_arguments)\n        except (ValueError, TypeError):\n            return raw_arguments\n    elif isinstance(raw_arguments, dict):\n        parsed = raw_arguments\n    else:\n        return str(raw_arguments)\n    if isinstance(parsed, dict) and isinstance(parsed.get("code"), str):\n        return parsed["code"]\n    return json.dumps(parsed, ensure_ascii=True)\n\n\ndef build_slice_digest(\n    messages: list[dict[str, Any]] | None,\n    head_marker: str | None,\n    *,\n    code_chars: int = DIGEST_CODE_CHARS,\n    result_chars: int = DIGEST_RESULT_CHARS,\n    reasoning_chars: int = DIGEST_REASONING_CHARS,\n    max_entries: int = DIGEST_MAX_ENTRIES,\n    max_chars: int = DIGEST_MAX_CHARS,\n) -> str:\n    """Bounded digest of the yielded slice\'s own messages.\n\n    The slice boundary is the LAST user message whose text starts with\n    ``head_marker`` (the slice\'s initial user prompt, recorded by the patched\n    ``_build_user_prompt``); everything after it is this slice\'s work. Returns\n    "" when there is nothing to carry (the empty-yield case).\n    """\n    if not messages or not head_marker:\n        return ""\n    boundary = None\n    for index in range(len(messages) - 1, -1, -1):\n        message = messages[index]\n        if not isinstance(message, dict):\n            continue\n        if str(message.get("role", "")).strip() != "user":\n            continue\n        if _message_text(message).startswith(head_marker):\n            boundary = index\n            break\n    if boundary is None:\n        return ""\n\n    entries: list[dict[str, Any]] = []\n    last_reasoning = ""\n    for message in messages[boundary + 1 :]:\n        if not isinstance(message, dict):\n            continue\n        role = str(message.get("role", "")).strip()\n        if role == "assistant":\n            reasoning = message.get("reasoning")\n            if isinstance(reasoning, str) and reasoning.strip():\n                last_reasoning = reasoning\n            for tool_call in message.get("tool_calls") or []:\n                entries.append(\n                    {\n                        "id": tool_call.get("id") if isinstance(tool_call, dict) else None,\n                        "code": _truncate(_tool_call_code(tool_call), code_chars),\n                        "result": None,\n                    }\n                )\n        elif role == "tool":\n            result_text = _truncate(_message_text(message) or str(message.get("content", "")), result_chars)\n            call_id = message.get("tool_call_id")\n            target = None\n            for entry in reversed(entries):\n                if entry["result"] is None and (entry["id"] == call_id or call_id in (None, "")):\n                    target = entry\n                    break\n            if target is None:\n                for entry in reversed(entries):\n                    if entry["result"] is None:\n                        target = entry\n                        break\n            if target is not None:\n                target["result"] = result_text\n            else:\n                entries.append({"id": call_id, "code": None, "result": result_text})\n\n    entries = entries[-max_entries:]\n    if not entries and not last_reasoning:\n        return ""\n\n    lines = [DIGEST_HEADER]\n    for number, entry in enumerate(entries, start=1):\n        if entry["code"] is not None:\n            lines.append(f"[{number}] python: {entry[\'code\']}")\n        else:\n            lines.append(f"[{number}] python: (code unavailable)")\n        if entry["result"] is not None:\n            lines.append(f"    result: {entry[\'result\']}")\n        else:\n            lines.append("    result: (no result captured before yield)")\n    if last_reasoning:\n        tail = last_reasoning.strip()[-reasoning_chars:]\n        lines.append(f"Last reasoning tail: {tail}")\n    lines.append(DIGEST_FOOTER)\n\n    digest = "\\n".join(lines)\n    while len(digest) > max_chars and len(lines) > 3:\n        # Drop the oldest entry line pair (after the header) until bounded.\n        del lines[1 : 3 if lines[2].startswith("    result:") else 2]\n        digest = "\\n".join(lines)\n    return digest\n\n\ndef install() -> str:\n    if not _carryover_enabled() and _slice_cap() <= 0:\n        return "yield_carryover: SKIP (YIELD_CARRYOVER=0 and YIELD_SLICE_CAP=0)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"yield_carryover: SKIP (tool_agent module missing: {exc!r})"\n\n    tool_agent_cls = getattr(agent_mod, "ToolAgent", None)\n    if tool_agent_cls is None:\n        return "yield_carryover: SKIP (missing ToolAgent)"\n    original_analyze = getattr(tool_agent_cls, "analyze", None)\n    original_persist = getattr(tool_agent_cls, "_persistent_history_messages", None)\n    original_build_prompt = getattr(tool_agent_cls, "_build_user_prompt", None)\n    if original_analyze is None:\n        return "yield_carryover: SKIP (ToolAgent.analyze missing)"\n    if original_persist is None:\n        return "yield_carryover: SKIP (ToolAgent._persistent_history_messages missing — wipe seam moved)"\n    if original_build_prompt is None:\n        return "yield_carryover: SKIP (ToolAgent._build_user_prompt missing — inject seam moved)"\n    turn_result = getattr(agent_mod, "AnalyzerTurnResult", None)\n    if turn_result is None or not hasattr(turn_result, "yielded_control"):\n        return "yield_carryover: SKIP (AnalyzerTurnResult.yielded_control missing)"\n    if getattr(original_analyze, "_yield_carryover_patched", False):\n        return "yield_carryover: SKIP (already applied)"\n\n    # --- digest source: stash the FULL slice conversation at the wipe seam ---\n\n    def persist_with_stash(self: Any, messages: list[dict[str, Any]], *args: Any, **kwargs: Any) -> Any:\n        try:\n            if _carryover_enabled() or _slice_cap() > 0:\n                self._yc_full_slice_messages = list(messages)\n        except Exception:  # noqa: BLE001 — stash must never break the turn\n            pass\n        return original_persist(self, messages, *args, **kwargs)\n\n    # --- injection point: the slice\'s initial user prompt ---\n\n    def build_prompt_with_injection(self: Any, *args: Any, **kwargs: Any) -> str:\n        prompt = original_build_prompt(self, *args, **kwargs)\n        try:\n            if not (_carryover_enabled() or _slice_cap() > 0):\n                return prompt\n            self._yc_slice_head = prompt[:_HEAD_MARKER_CHARS]\n            pending = getattr(self, "_yc_pending_inject", None)\n            if not isinstance(pending, dict):\n                return prompt\n            self._yc_pending_inject = None  # consume: injected exactly once\n            digest = pending.get("digest")\n            if pending.get("final") and FINAL_SLICE_INSTRUCTION not in prompt:\n                prompt = f"{FINAL_SLICE_INSTRUCTION}\\n\\n{prompt}"\n            if digest and DIGEST_HEADER not in prompt:\n                prompt = f"{prompt}\\n\\n{digest}"\n        except Exception:  # noqa: BLE001 — injection failure => stock prompt\n            pass\n        return prompt\n\n    # --- keep exactly ONE injection alive: scrub old blocks from history ---\n\n    def _strip_injections(text: str) -> str:\n        index = text.find(DIGEST_HEADER)\n        if index != -1:\n            start = text.rfind("\\n\\n", 0, index)\n            start = start if start != -1 else index\n            end = text.find(DIGEST_FOOTER, index)\n            end = end + len(DIGEST_FOOTER) if end != -1 else len(text)\n            text = text[:start] + text[end:]\n        if text.startswith(FINAL_SLICE_INSTRUCTION):\n            text = text[len(FINAL_SLICE_INSTRUCTION) :].lstrip("\\n")\n        return text\n\n    def _scrub_history(self: Any) -> None:\n        """Remove digest/final blocks from carried user messages so a resumed\n        slice never sees two copies (and the token cost never compounds)."""\n        history = getattr(self, "_history_messages", None)\n        if not isinstance(history, list):\n            return\n        for message in history:\n            if not isinstance(message, dict) or str(message.get("role", "")).strip() != "user":\n                continue\n            content = message.get("content")\n            if isinstance(content, str) and (DIGEST_HEADER in content or content.startswith(FINAL_SLICE_INSTRUCTION)):\n                message["content"] = _strip_injections(content)\n            elif isinstance(content, list):\n                for part in content:\n                    if isinstance(part, dict) and part.get("type") == "text":\n                        text = str(part.get("text", ""))\n                        if DIGEST_HEADER in text or text.startswith(FINAL_SLICE_INSTRUCTION):\n                            part["text"] = _strip_injections(text)\n\n    # --- turn/slice tracking around analyze ---\n\n    def _pre_analyze(self: Any, state_path: Any, kwargs: dict[str, Any]) -> None:\n        _scrub_history(self)\n        analysis_step = kwargs.get("analysis_step")\n        key = (str(state_path), analysis_step)\n        previous_key = getattr(self, "_yc_key", None)\n        slices_done = int(getattr(self, "_yc_slices", 0) or 0)\n        is_resume = analysis_step is not None and key == previous_key and slices_done >= 1\n        if not is_resume:\n            self._yc_key = key\n            self._yc_slices = 0\n            self._yc_digest = None\n            self._yc_pending_inject = None\n            return\n        slice_number = slices_done + 1\n        inject: dict[str, Any] = {}\n        if _carryover_enabled() and getattr(self, "_yc_digest", None):\n            inject["digest"] = self._yc_digest\n        cap = _slice_cap()\n        if cap > 0 and slice_number >= cap:\n            inject["final"] = True\n        self._yc_pending_inject = inject or None\n\n    def _post_analyze(self: Any, result: Any) -> None:\n        self._yc_pending_inject = None  # never let an unconsumed block leak forward\n        yielded = result is not None and bool(getattr(result, "yielded_control", False))\n        acted = result is not None and bool(getattr(result, "step_executed", False))\n        if yielded and not acted:\n            self._yc_slices = int(getattr(self, "_yc_slices", 0) or 0) + 1\n            if _carryover_enabled():\n                digest = build_slice_digest(\n                    getattr(self, "_yc_full_slice_messages", None),\n                    getattr(self, "_yc_slice_head", None),\n                )\n                if digest:\n                    # Empty yielded slices keep the previous digest alive.\n                    self._yc_digest = digest\n            return\n        self._yc_key = None\n        self._yc_slices = 0\n        self._yc_digest = None\n\n    def analyze_with_carryover(self: Any, state_path: Any, action_num: int, *args: Any, **kwargs: Any) -> Any:\n        enabled = _carryover_enabled() or _slice_cap() > 0\n        if enabled:\n            try:\n                _pre_analyze(self, state_path, kwargs)\n            except Exception:  # noqa: BLE001\n                pass\n        # Never guard the inner call: an inner crash must propagate as stock.\n        result = original_analyze(self, state_path, action_num, *args, **kwargs)\n        if enabled:\n            try:\n                _post_analyze(self, result)\n            except Exception:  # noqa: BLE001\n                pass\n        return result\n\n    analyze_with_carryover._yield_carryover_patched = True  # type: ignore[attr-defined]\n    tool_agent_cls._persistent_history_messages = persist_with_stash\n    tool_agent_cls._build_user_prompt = build_prompt_with_injection\n    tool_agent_cls.analyze = analyze_with_carryover\n    # Expose originals for tests / debugging (single-namespace: methods are\n    # only ever resolved via ``self.`` — no by-name imports of these symbols).\n    install.originals = {  # type: ignore[attr-defined]\n        "analyze": original_analyze,\n        "_persistent_history_messages": original_persist,\n        "_build_user_prompt": original_build_prompt,\n    }\n    return "yield_carryover: OK"\n'

import importlib.util as _ilu


def _load_graft(name, source):
    path = WORKING_DIR / (name + ".py")
    path.write_text(source, encoding="utf-8")
    spec = _ilu.spec_from_file_location(name, path)
    mod = _ilu.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


# Install ORDER: effort first (the baseline), carryover second (the ladder
# variable wraps on top). Seams are disjoint except ToolAgent.analyze, where
# carryover deliberately wraps effort's wrapper.
_effort = _load_graft("graft_effort", _EFFORT_SOURCE)
_effort_status = _effort.install()
print("[effort]", _effort_status)
assert _effort_status == "effort_medium: OK", (
    "baseline graft must be live, got: " + repr(_effort_status)
)
_carry = _load_graft("graft_carryover", _CARRYOVER_SOURCE)
_carry_status = _carry.install()
print("[carryover]", _carry_status)
# A smoke that silently measures the baseline alone is worse than one that
# dies: hard gate on the ladder variable.
assert _carry_status == "yield_carryover: OK", (
    "ladder-variable graft must be live, got: " + repr(_carry_status)
)


In [ ]:
# Carryover smoke telemetry — the reads this smoke exists for:
# yield-resume events; slices per turn (max + distribution); byte-identical
# duplicate snippets within turns (corpus baseline: 71 duplicates, worst turn
# 35 slices); history_messages count on resumed slices (corpus collapse: 2-4);
# digest injections (exactly one alive per resumed slice — multi-digest
# counters must stay 0); effort baseline proof (reasoning_effort=medium
# payloads) and dead completions (effort-smoke read continuity).
# Wraps the ALREADY-GRAFTED seams — counting sits outside both grafts.
import threading as _tel_threading
from collections import Counter as _tel_Counter

import graft_carryover as _g_carry
import graft_effort as _g_eff
from inference.agent import tool_agent as _tel_ta
from inference.utils import openai_compat as _tel_oc

TELEMETRY = {
    "requests": 0,
    "payloads": 0,
    "payloads_effort_medium": 0,
    "dead_completions": 0,
    "reasoning_chars": 0,
    "tool_call_completions": 0,
    "finish_reasons": {},
    "analyze_calls": 0,
    "turns_completed": 0,
    "yield_noaction_events": 0,
    "resume_slices": 0,
    "slices_per_turn_dist": {},
    "max_slices_per_turn": 0,
    "duplicate_snippets_total": 0,
    "worst_turn_duplicates": 0,
    "turns_with_duplicates": 0,
    "history_len_on_resume": [],
    "history_len_on_resume_stats": {"n": 0, "min": None, "max": None, "sum": 0},
    "digest_prompts": 0,
    "final_prompts": 0,
    "multi_digest_prompts": 0,
    "requests_with_digest": 0,
    "multi_digest_requests": 0,
    "max_digest_blocks_in_request": 0,
}
_tel_lock = _tel_threading.Lock()
_TEL_PATH = WORKING_DIR / "carryover_telemetry.json"
_TEL_AGENTS = {}  # id(agent) -> agent, so open turns can be flushed at report time

# --- effort baseline proof: payloads actually carrying reasoning_effort=medium ---
_tel_inner_build = _tel_oc.build_chat_payload
assert getattr(_tel_inner_build, "_effort_medium_patched", False), "effort graft must be installed first"


def _tel_build(*args, **kwargs):
    payload = _tel_inner_build(*args, **kwargs)
    try:
        effort = (payload.get("chat_template_kwargs") or {}).get("reasoning_effort")
        with _tel_lock:
            TELEMETRY["payloads"] += 1
            if effort == "medium":
                TELEMETRY["payloads_effort_medium"] += 1
    except Exception:  # noqa: BLE001 — telemetry must never break a request
        pass
    return payload


_tel_build._effort_medium_patched = True  # keep graft idempotence marker intact
_tel_oc.build_chat_payload = _tel_build
_tel_ta.build_chat_payload = _tel_build

# --- digest injection read: the slice's initial user prompt, post-graft ---
_tel_inner_prompt = _tel_ta.ToolAgent._build_user_prompt


def _tel_prompt(self, *args, **kwargs):
    prompt = _tel_inner_prompt(self, *args, **kwargs)
    try:
        n_digest = prompt.count(_g_carry.DIGEST_HEADER)
        with _tel_lock:
            if n_digest:
                TELEMETRY["digest_prompts"] += 1
                if n_digest > 1:
                    TELEMETRY["multi_digest_prompts"] += 1
            if prompt.startswith(_g_carry.FINAL_SLICE_INSTRUCTION):
                TELEMETRY["final_prompts"] += 1  # must stay 0: YIELD_SLICE_CAP=0
    except Exception:  # noqa: BLE001
        pass
    return prompt


_tel_ta.ToolAgent._build_user_prompt = _tel_prompt

# --- wire-level: digest blocks per request + dead completions + turn snippets ---
_tel_inner_chat = _tel_ta.ToolAgent._chat_completion


def _tel_record(self, messages, result):
    digest_blocks = 0
    try:
        for message in messages or []:
            if isinstance(message, dict):
                digest_blocks += _g_carry._message_text(message).count(_g_carry.DIGEST_HEADER)
    except Exception:  # noqa: BLE001
        digest_blocks = -1
    message = getattr(result, "message", None)
    message = message if isinstance(message, dict) else {}
    finish = str(getattr(result, "finish_reason", "") or "")
    try:
        reasoning = _tel_ta._extract_reasoning_text(message)
    except Exception:  # noqa: BLE001
        reasoning = ""
    dead = _g_eff._is_dead_completion(result, _tel_ta)
    codes = []
    for tool_call in message.get("tool_calls") or []:
        try:
            codes.append(_g_carry._tool_call_code(tool_call))
        except Exception:  # noqa: BLE001
            pass
    if codes:
        bucket = getattr(self, "_tel_turn_codes", None)
        if bucket is None:
            bucket = []
            self._tel_turn_codes = bucket
        bucket.extend(codes)
    with _tel_lock:
        TELEMETRY["requests"] += 1
        TELEMETRY["finish_reasons"][finish] = TELEMETRY["finish_reasons"].get(finish, 0) + 1
        TELEMETRY["reasoning_chars"] += len(reasoning or "")
        if message.get("tool_calls"):
            TELEMETRY["tool_call_completions"] += 1
        if dead:
            TELEMETRY["dead_completions"] += 1
        if digest_blocks > 0:
            TELEMETRY["requests_with_digest"] += 1
            if digest_blocks > TELEMETRY["max_digest_blocks_in_request"]:
                TELEMETRY["max_digest_blocks_in_request"] = digest_blocks
            if digest_blocks > 1:
                TELEMETRY["multi_digest_requests"] += 1
        n = TELEMETRY["requests"]
        snapshot = json.dumps(TELEMETRY, indent=1)
    if n % 20 == 0:
        try:
            _TEL_PATH.write_text(snapshot, encoding="utf-8")
        except Exception:  # noqa: BLE001
            pass
        print(
            f"[carryover-telemetry] req={n} resumes={TELEMETRY['resume_slices']} "
            f"yields={TELEMETRY['yield_noaction_events']} "
            f"dups={TELEMETRY['duplicate_snippets_total']} "
            f"max_slices={TELEMETRY['max_slices_per_turn']} "
            f"digest_prompts={TELEMETRY['digest_prompts']} "
            f"effort_payloads={TELEMETRY['payloads_effort_medium']}/{TELEMETRY['payloads']}",
            flush=True,
        )


def _tel_chat(self, messages, *args, **kwargs):
    result = _tel_inner_chat(self, messages, *args, **kwargs)
    try:
        _tel_record(self, messages, result)
    except Exception:  # noqa: BLE001
        pass
    return result


_tel_ta.ToolAgent._chat_completion = _tel_chat

# --- turn/slice accounting around analyze (outermost wrapper) ---
_tel_inner_analyze = _tel_ta.ToolAgent.analyze
assert getattr(_tel_inner_analyze, "_yield_carryover_patched", False), "carryover graft must be installed first"


def _tel_flush_turn(self):
    codes = getattr(self, "_tel_turn_codes", None) or []
    slices = int(getattr(self, "_tel_turn_slices", 0) or 0)
    if slices <= 0 and not codes:
        return
    counts = _tel_Counter(codes)
    dups = sum(c - 1 for c in counts.values() if c > 1)
    with _tel_lock:
        TELEMETRY["turns_completed"] += 1
        key = str(slices)
        TELEMETRY["slices_per_turn_dist"][key] = TELEMETRY["slices_per_turn_dist"].get(key, 0) + 1
        if slices > TELEMETRY["max_slices_per_turn"]:
            TELEMETRY["max_slices_per_turn"] = slices
        TELEMETRY["duplicate_snippets_total"] += dups
        if dups:
            TELEMETRY["turns_with_duplicates"] += 1
            if dups > TELEMETRY["worst_turn_duplicates"]:
                TELEMETRY["worst_turn_duplicates"] = dups
    self._tel_turn_codes = []
    self._tel_turn_slices = 0


def _tel_analyze(self, state_path, action_num, *args, **kwargs):
    try:
        _TEL_AGENTS[id(self)] = self
        analysis_step = kwargs.get("analysis_step")
        key = (str(state_path), analysis_step)
        # Mirror the graft's resume test against the graft's OWN pre-call state
        # (telemetry is outermost, so _yc_* still reflect the previous slice).
        is_resume = (
            analysis_step is not None
            and key == getattr(self, "_yc_key", None)
            and int(getattr(self, "_yc_slices", 0) or 0) >= 1
        )
        if is_resume:
            hist = getattr(self, "_history_messages", None)
            hist_len = len(hist) if isinstance(hist, list) else -1
            with _tel_lock:
                TELEMETRY["resume_slices"] += 1
                stats = TELEMETRY["history_len_on_resume_stats"]
                stats["n"] += 1
                stats["sum"] += max(hist_len, 0)
                if stats["min"] is None or hist_len < stats["min"]:
                    stats["min"] = hist_len
                if stats["max"] is None or hist_len > stats["max"]:
                    stats["max"] = hist_len
                if len(TELEMETRY["history_len_on_resume"]) < 400:
                    TELEMETRY["history_len_on_resume"].append(hist_len)
        else:
            _tel_flush_turn(self)  # previous turn left open on this agent
        self._tel_turn_slices = int(getattr(self, "_tel_turn_slices", 0) or 0) + 1
    except Exception:  # noqa: BLE001
        pass
    # Never guard the inner call: an inner crash must propagate as stock.
    result = _tel_inner_analyze(self, state_path, action_num, *args, **kwargs)
    try:
        with _tel_lock:
            TELEMETRY["analyze_calls"] += 1
        yielded = result is not None and bool(getattr(result, "yielded_control", False))
        acted = result is not None and bool(getattr(result, "step_executed", False))
        if yielded and not acted:
            with _tel_lock:
                TELEMETRY["yield_noaction_events"] += 1
        else:
            _tel_flush_turn(self)
    except Exception:  # noqa: BLE001
        pass
    return result


_tel_analyze._yield_carryover_patched = True  # keep graft idempotence marker intact
_tel_ta.ToolAgent.analyze = _tel_analyze
print("[carryover-telemetry] counters installed (corpus baseline: 71 dup snippets, worst turn 35 slices, history collapse 2-4)")


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
# ---- carryover-smoke final report (grep for CARRYOVER SMOKE / GAME / TELEMETRY) ----
for _agent in list(_TEL_AGENTS.values()):
    try:
        _tel_flush_turn(_agent)  # close any turn left open at game end
    except Exception:  # noqa: BLE001
        pass

print("=" * 72)
print("CARRYOVER SMOKE RESULTS (YIELD_CARRYOVER=1, YIELD_SLICE_CAP=0, EFFORT_MEDIUM=1, EFFORT_DEAD_RETRY=0)")
games_out = []
for game_run in bm.game_runs:
    actions = sum(game_run.actions_per_level) if game_run.actions_per_level else len(game_run.history)
    row = {
        "game_id": game_run.game_id,
        "state": game_run.state,
        "levels_completed": game_run.levels_completed,
        "number_of_levels": game_run.number_of_levels,
        "final_score": game_run.final_score,
        "actions": actions,
        "wallclock_s": game_run.final_wallclock_seconds,
    }
    games_out.append(row)
    print(
        f"GAME {row['game_id']}: state={row['state']} "
        f"levels={row['levels_completed']}/{row['number_of_levels']} "
        f"final_score={row['final_score']} actions={row['actions']} "
        f"wallclock_s={row['wallclock_s']}"
    )

with _tel_lock:
    tel = json.loads(json.dumps(TELEMETRY))
hist_stats = tel["history_len_on_resume_stats"]
hist_mean = (hist_stats["sum"] / hist_stats["n"]) if hist_stats["n"] else None
print(
    f"TELEMETRY analyze_calls={tel['analyze_calls']} turns_completed={tel['turns_completed']} "
    f"yield_noaction_events={tel['yield_noaction_events']} resume_slices={tel['resume_slices']}"
)
print(
    f"TELEMETRY slices_per_turn max={tel['max_slices_per_turn']} "
    f"(corpus worst 35) dist={tel['slices_per_turn_dist']}"
)
print(
    f"TELEMETRY duplicate_snippets_total={tel['duplicate_snippets_total']} "
    f"(corpus baseline 71) worst_turn={tel['worst_turn_duplicates']} "
    f"turns_with_duplicates={tel['turns_with_duplicates']}"
)
print(
    f"TELEMETRY history_len_on_resume min={hist_stats['min']} max={hist_stats['max']} "
    f"mean={hist_mean} n={hist_stats['n']} (corpus collapse: 2-4)"
)
print(
    f"TELEMETRY digest_prompts={tel['digest_prompts']} final_prompts={tel['final_prompts']} "
    f"multi_digest_prompts={tel['multi_digest_prompts']} "
    f"requests_with_digest={tel['requests_with_digest']} "
    f"multi_digest_requests={tel['multi_digest_requests']} "
    f"max_digest_blocks_in_request={tel['max_digest_blocks_in_request']}"
)
print(
    f"TELEMETRY requests={tel['requests']} payloads={tel['payloads']} "
    f"effort_medium_payloads={tel['payloads_effort_medium']} "
    f"dead_completions={tel['dead_completions']} "
    f"reasoning_chars_total={tel['reasoning_chars']} "
    f"tool_call_completions={tel['tool_call_completions']}"
)
print("TELEMETRY finish_reasons=", tel["finish_reasons"])

_levels = {row["game_id"].split("-")[0]: row["levels_completed"] for row in games_out}
_no_regression = _levels.get("vc33", 0) >= 2 and _levels.get("tn36", 0) >= 1
print(
    "PASS BARS: dup snippets + max slices/turn collapse vs corpus (71 dups / worst 35); "
    "no level regression vs effort-smoke (vc33>=2, tn36>=1). "
    f"level_bar_met={_no_regression} "
    f"observed dups={tel['duplicate_snippets_total']} max_slices={tel['max_slices_per_turn']}"
)

_results = {
    "arm": "carryover-smoke",
    "flags": {
        "YIELD_CARRYOVER": "1",
        "YIELD_SLICE_CAP": "0",
        "EFFORT_MEDIUM": "1",
        "EFFORT_DEAD_RETRY": "0",
    },
    "games": games_out,
    "telemetry": tel,
    "level_bar_met": _no_regression,
    "comparators": {
        "effort_smoke_0823": {
            "vc33": {"levels": 2, "score": 4.7319},
            "tn36": {"levels": 1, "score": 1.5232},
            "ft09": {"levels": 0, "score": 0.0},
            "sk48": {"levels": 0, "score": 0.0},
        },
        "corpus_duplicate_snippets": 71,
        "corpus_worst_turn_slices": 35,
        "corpus_history_collapse": "2-4",
    },
}
(WORKING_DIR / "carryover_smoke_results.json").write_text(
    json.dumps(_results, indent=1), encoding="utf-8"
)
print("wrote", WORKING_DIR / "carryover_smoke_results.json")
